In [ ]:
import pandas as pd
import numpy as np
import torch as tc
import torch.nn as nn
import math as mt

from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader

In [ ]:
df = pd.read_csv("ranking_dataset.csv")

value_cols = [f"val_{i}" for i in range(10)]
rank_cols = [f"rank_{i}" for i in range(10)]

In [ ]:
class rank_data(Dataset):
    def __init__(self, dataframe):

        self.X = tc.tensor(dataframe[value_cols].values,dtype=tc.float32)
        self.y = tc.tensor(dataframe[rank_cols].values,dtype=tc.long)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

In [ ]:
train_df, temp_df = train_test_split(df,test_size=0.2,random_state=42)
val_df, test_df = train_test_split(temp_df,test_size=0.50,random_state=42)

train_data = rank_data(train_df)
val_data = rank_data(val_df)
test_data = rank_data(test_df)

In [ ]:
train_loader = DataLoader(train_data,batch_size=64,shuffle=True)
val_loader = DataLoader(val_data,batch_size=64,shuffle=False)
test_loader = DataLoader(test_data,batch_size=64,shuffle=False)

In [ ]:
class PositionalEncoding(nn.Module):

    def __init__(self, d_model, max_len=5000):
        super().__init__()
        pe = tc.zeros(max_len, d_model)
        
        position = tc.arange(0,max_len,dtype=tc.float).unsqueeze(1)
        div_term = tc.exp(tc.arange(0,d_model,2).float()*(-mt.log(10000.0) / d_model))
        
        pe[:, 0::2] = tc.sin(position * div_term)
        pe[:, 1::2] = tc.cos(position * div_term)
        self.register_buffer("pe",pe.unsqueeze(0))
    
    def forward(self, x):
        return x + self.pe[:, :x.size(1)]

In [ ]:
class Attention(nn.Module):

    def __init__(self, d_model, num_heads):
        super().__init__()

        assert d_model % num_heads == 0
        self.d_model = d_model
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads

        self.Wq = nn.Linear(d_model, d_model)
        self.Wk = nn.Linear(d_model, d_model)
        self.Wv = nn.Linear(d_model, d_model)

        self.fc = nn.Linear(d_model, d_model)

    def forward(self, x):
        batch_size = x.size(0)
        Q = self.Wq(x)
        K = self.Wk(x)
        V = self.Wv(x)
        
        Q = Q.view(batch_size,-1,self.num_heads,self.head_dim).transpose(1, 2)
        K = K.view(batch_size,-1,self.num_heads,self.head_dim).transpose(1, 2)
        V = V.view(batch_size,-1,self.num_heads,self.head_dim).transpose(1, 2)
        
        scores = tc.matmul(Q,K.transpose(-2, -1)) / mt.sqrt(self.head_dim)

        attention = tc.softmax(scores,dim=-1)
        self.attention_weights = attention
        out = tc.matmul(attention,V)
        out = out.transpose(1,2).contiguous()
        out = out.view(batch_size,-1,self.d_model)

        return self.fc(out)

In [ ]:
class Forward(nn.Module):

    def __init__(self, d_model, ff_dim):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(d_model, ff_dim),nn.ReLU(),nn.Linear(ff_dim, d_model))

    def forward(self, x):
        return self.net(x)

In [ ]:
class Transformer(nn.Module):

    def __init__(self, d_model, num_heads, ff_dim):
        super().__init__()

        self.attn = Attention(d_model,num_heads)
        self.norm1 = nn.LayerNorm(d_model)
        self.ff = Forward(d_model,ff_dim)
        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, x):
        attn_out = self.attn(x)
        x = self.norm1(x + attn_out)
        ff_out = self.ff(x)
        x = self.norm2(x + ff_out)

        return x

In [ ]:
class Ranker(nn.Module):
    
    def __init__(self,head=4):
        super().__init__()
        self.d_model = 64
        self.embedding = nn.Linear(1,self.d_model)
        self.pos_encoding = PositionalEncoding(self.d_model,max_len=10)
        self.encoder1 = Transformer(d_model=64,num_heads=head,ff_dim=128)
        self.encoder2 = Transformer(d_model=64,num_heads=head,ff_dim=128)
        self.classifier = nn.Linear(64,10)

    def forward(self, x):
        x = x.unsqueeze(-1)
        x = self.embedding(x)
        x = self.pos_encoding(x)
        x = self.encoder1(x)
        x = self.encoder2(x)
        x = self.classifier(x)
        return x

In [ ]:
def evaluate(model, loader):
    model.eval()
    correct = 0
    total = 0
    with tc.no_grad():
        for x, y in loader:
            logits = model(x)
            preds = logits.argmax(dim=-1)
            correct += (preds == y).sum().item()
            total += y.numel()
    return correct / total

In [ ]:
def train_model(model, train_loader, val_loader, epochs=20,lrate=1e-3):
    criterion = nn.CrossEntropyLoss()
    optimizer = tc.optim.Adam(model.parameters(),lr=lrate)
    for epoch in range(epochs):
        model.train()
        running_loss = 0
        for x, y in train_loader:
            optimizer.zero_grad()
            logits = model(x)
            loss = criterion(logits.view(-1, 10),y.view(-1)) 
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
        train_acc = evaluate(model, train_loader)
        val_acc = evaluate(model, val_loader)
        print(
            f"Epoch {epoch+1}: "
            f"Loss={running_loss/len(train_loader):.4f} | "
            f"Train={train_acc:.4f} | "
            f"Val={val_acc:.4f}"
        )
    return model

### Experiment 1: Effect of Number of Attention Heads

The number of attention heads was increased from 4 to 8 while keeping the rest of the Transformer architecture unchanged.

#### Observation

The 4-head model achieved a test accuracy of 97.31%, while the 8-head model achieved 97.38%. The improvement was only 0.07%.

#### Conclusion

The ranking task can already be modeled effectively using four attention heads. Increasing the number of heads provides only a marginal improvement, suggesting that additional attention heads do not contribute significantly to performance on this task.

In [ ]:
model_4 = Ranker(head=4)
model_8 = Ranker(head=8)
print("Head = 4")
train_model(model_4,train_loader,val_loader)
transformer_test_acc_4 = evaluate(model_4,test_loader)
print("\nHead = 8")
train_model(model_8,train_loader,val_loader)
transformer_test_acc_8 = evaluate(model_8,test_loader)

print("Head=4 => Transformer Test Acc:",transformer_test_acc_4)
print("Head=8 => Transformer Test Acc:",transformer_test_acc_8)

### Experiment 2: Positional Encoding Ablation

Positional encoding was removed from the Transformer while keeping the rest of the architecture unchanged.

#### Observation

The Transformer with positional encoding achieved a test accuracy of 97.34%, whereas the model without positional encoding achieved only 73.36%. Training also became less stable, with a significant drop in accuracy during the final epochs.

#### Conclusion

Positional encoding plays an important role in the ranking task. Although ranking depends on numerical comparisons, positional information helps the Transformer distinguish elements and learn more reliable relationships between them. Removing positional encoding resulted in a large decrease in performance and training stability.

In [ ]:
class RankerNoPE(nn.Module):

    def __init__(self,head=4):
        super().__init__()
        self.d_model = 64
        self.embedding = nn.Linear(1,self.d_model)
        self.encoder1 = Transformer(d_model=64,num_heads=head,ff_dim=128)
        self.encoder2 = Transformer(d_model=64,num_heads=head,ff_dim=128)
        self.classifier = nn.Linear(64,10)

    def forward(self, x):
        x = x.unsqueeze(-1)
        x = self.embedding(x)
        x = self.encoder1(x)
        x = self.encoder2(x)
        x = self.classifier(x)
        return x

In [ ]:
nope_model = RankerNoPE()

train_model(nope_model,train_loader,val_loader)
nope_acc = evaluate(nope_model,test_loader)
print("No Positional Encoding Test Acc:",nope_acc)

### Experiment 3: Continuous Representation vs Embedding Representation

This experiment compares two ways of representing numbers inside the Transformer. The original model uses a linear projection on raw numerical values, while the second model treats each number as a categorical token using an embedding layer.

#### Observation

The continuous representation achieved a test accuracy of 97.34%, whereas the embedding-based representation achieved 80.25%.

Training was slower for the embedding model and the final accuracy remained significantly lower.

#### Conclusion

The continuous representation is more suitable for the ranking task because it preserves numerical relationships between values. Numbers such as 5 and 6 remain naturally related after projection. In contrast, the embedding layer treats numbers as independent symbols and must learn these relationships from data, making the task more difficult.

In [ ]:
class RankerEmb(nn.Module):

    def __init__(self):
        super().__init__()
        self.d_model = 64
        self.embedding = nn.Embedding(1001,self.d_model)
        self.pos_encoding = PositionalEncoding(self.d_model,max_len=10)
        self.encoder1 = Transformer(d_model=64,num_heads=4,ff_dim=128)
        self.encoder2 = Transformer(d_model=64,num_heads=4,ff_dim=128)
        self.classifier = nn.Linear(64,10)

    def forward(self, x):
        x = x.long()
        x = self.embedding(x)
        x = self.pos_encoding(x)
        x = self.encoder1(x)
        x = self.encoder2(x)
        x = self.classifier(x)

        return x

In [ ]:
emb_model = RankerEmb()

train_model(emb_model,train_loader,val_loader)
emb_acc = evaluate(emb_model,test_loader)
print("Embedding Test Acc:",emb_acc)

### Experiment 4.1: Z-Score Normalization

In this experiment, the input values were normalized using Z-score normalization before training. The goal was to investigate whether scaling the numerical inputs improves convergence and overall performance.

#### Observation

The model achieved a test accuracy of 97.20%, which is very close to the 97.34% obtained using raw inputs. Training was smooth and converged steadily throughout the epochs.

#### Conclusion

Z-score normalization did not significantly change the final performance of the Transformer. This suggests that the model is already capable of handling raw numerical values effectively. However, normalization produced stable training behavior and may still be useful when working with datasets containing larger variations in scale.

In [ ]:
df_z = df.copy()

for col in value_cols:
    mean = df_z[col].mean()
    std = df_z[col].std()
    df_z[col] = (df_z[col] - mean) / std

In [ ]:
train_df_z, temp_df_z = train_test_split(df_z,test_size=0.2,random_state=42)
val_df_z, test_df_z = train_test_split(temp_df_z,test_size=0.5,random_state=42)

train_data_z = rank_data(train_df_z)
val_data_z = rank_data(val_df_z)
test_data_z = rank_data(test_df_z)

train_loader_z = DataLoader(train_data_z,batch_size=64,shuffle=True)
val_loader_z = DataLoader(val_data_z,batch_size=64,shuffle=False)
test_loader_z = DataLoader(test_data_z,batch_size=64,shuffle=False)

In [ ]:
z_model = Ranker()

train_model(z_model,train_loader_z,val_loader_z)
z_acc = evaluate(z_model,test_loader_z)
print("Z Score Norm Test Acc:",z_acc)

In [ ]:
df_mm = df.copy()

for col in value_cols:
    mn = df_mm[col].min()
    mx = df_mm[col].max()
    df_mm[col] = (df_mm[col] - mn) / (mx - mn)

### Experiment 4.2: Min-Max Scaling

In this experiment, the input values were scaled to the range [0, 1] using Min-Max normalization. The objective was to determine whether restricting the numerical range while preserving relative ordering improves Transformer performance.

#### Observation

The model achieved a test accuracy of 97.80%, which was higher than both the raw input model (97.34%) and the Z-score normalized model (97.20%). Training was also stable and converged smoothly throughout the experiment.

#### Conclusion

Min-Max scaling produced the best performance among the normalization methods tested so far. Since ranking depends primarily on relative ordering, preserving the relative positions of values while keeping them within a fixed range appears to help the Transformer learn more effectively.

In [ ]:
train_df_mm, temp_df_mm = train_test_split(df_mm,test_size=0.2,random_state=42)
val_df_mm, test_df_mm = train_test_split(temp_df_mm,test_size=0.5,random_state=42)  

train_data_mm = rank_data(train_df_mm)
val_data_mm = rank_data(val_df_mm)
test_data_mm = rank_data(test_df_mm)

train_loader_mm = DataLoader(train_data_mm,batch_size=64,shuffle=True)
val_loader_mm = DataLoader(val_data_mm,batch_size=64,shuffle=False)
test_loader_mm = DataLoader(test_data_mm,batch_size=64,shuffle=False)

In [ ]:
mm_model = Ranker()

train_model(mm_model,train_loader_mm,val_loader_mm)
mm_acc = evaluate(mm_model,test_loader_mm)

print("Min-Max Test Acc:",mm_acc)

### Experiment 4.3: Sequence-Wise Normalization

In this experiment, each sequence was normalized independently using its own mean and standard deviation. This removes absolute scale information while preserving the relative relationships between values within a sequence.

#### Observation

The model achieved a test accuracy of 98.34%, which was the highest among all normalization methods tested. Training also converged very quickly, reaching over 86% accuracy in the first epoch and continuing to improve steadily throughout training.

#### Conclusion

Sequence-wise normalization produced the best performance because ranking depends on relative ordering rather than absolute numerical values. By removing scale information and preserving only the relationships between elements within a sequence, the model was able to learn the ranking task more effectively and achieve the highest accuracy.

In [ ]:
class rank_data_seqnorm(Dataset):

    def __init__(self, dataframe):

        X = dataframe[value_cols].values.astype(np.float32)

        mean = X.mean(axis=1, keepdims=True)
        std = X.std(axis=1, keepdims=True)

        X = (X - mean) / (std + 1e-8)

        self.X = tc.tensor(X,dtype=tc.float32)
        self.y = tc.tensor(dataframe[rank_cols].values,dtype=tc.long)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

In [ ]:
train_data_seq = rank_data_seqnorm(train_df)
val_data_seq = rank_data_seqnorm(val_df)
test_data_seq = rank_data_seqnorm(test_df)

train_loader_seq = DataLoader(train_data_seq,batch_size=64,shuffle=True)
val_loader_seq = DataLoader(val_data_seq,batch_size=64,shuffle=False)
test_loader_seq = DataLoader(test_data_seq,batch_size=64,shuffle=False)

In [67]:
seq_model = Ranker()

train_model(seq_model,train_loader_seq,val_loader_seq)
seq_acc = evaluate(seq_model,test_loader_seq)

print("Sequence-Wise Test Acc:",seq_acc)

tc.save(seq_model.state_dict(), "best_transformer_ranker.pth")

Epoch 1: Loss=0.9435 | Train=0.8504 | Val=0.8528
Epoch 2: Loss=0.2970 | Train=0.9238 | Val=0.9220
Epoch 3: Loss=0.1788 | Train=0.9414 | Val=0.9401
Epoch 4: Loss=0.1343 | Train=0.9589 | Val=0.9576
Epoch 5: Loss=0.1106 | Train=0.9684 | Val=0.9690
Epoch 6: Loss=0.0934 | Train=0.9612 | Val=0.9582
Epoch 7: Loss=0.0849 | Train=0.9682 | Val=0.9665
Epoch 8: Loss=0.0855 | Train=0.9751 | Val=0.9735
Epoch 9: Loss=0.0817 | Train=0.9717 | Val=0.9692
Epoch 10: Loss=0.0730 | Train=0.9728 | Val=0.9702
Epoch 11: Loss=0.0716 | Train=0.9779 | Val=0.9769
Epoch 12: Loss=0.0703 | Train=0.9772 | Val=0.9749
Epoch 13: Loss=0.0616 | Train=0.9689 | Val=0.9669
Epoch 14: Loss=0.0606 | Train=0.9819 | Val=0.9802
Epoch 15: Loss=0.0599 | Train=0.9787 | Val=0.9766
Epoch 16: Loss=0.0620 | Train=0.9738 | Val=0.9717
Epoch 17: Loss=0.0779 | Train=0.9787 | Val=0.9765
Epoch 18: Loss=0.0568 | Train=0.9750 | Val=0.9733
Epoch 19: Loss=0.0568 | Train=0.9827 | Val=0.9809
Epoch 20: Loss=0.0581 | Train=0.9633 | Val=0.9623
Sequence-

### Experiment 5: Effect of Transformer Depth

This experiment investigates how the number of Transformer encoder layers affects performance on the ranking task. The number of encoder layers was reduced from two to one while keeping all other settings unchanged.

#### Observation

The two-layer Transformer achieved a test accuracy of 95.17%, whereas the one-layer Transformer achieved only 50.74%.

The one-layer model struggled to learn the ranking task and training remained unstable throughout the experiment. In contrast, the two-layer model converged successfully and achieved much higher accuracy.

#### Conclusion

The second encoder layer plays an important role in improving Transformer performance on the ranking task. Multiple encoder layers allow the model to repeatedly refine information obtained through attention and perform more effective comparisons between elements. As a result, the deeper Transformer achieved significantly better ranking accuracy.

In [ ]:
class Ranker1Layer(nn.Module):

    def __init__(self):
        super().__init__()

        self.d_model = 64
        self.embedding = nn.Linear(1,self.d_model)
        self.pos_encoding = PositionalEncoding(self.d_model,max_len=10)
        self.encoder1 = Transformer(d_model=64,num_heads=4,ff_dim=128)
        self.classifier = nn.Linear(64,10)

    def forward(self, x):
        x = x.unsqueeze(-1)
        x = self.embedding(x)
        x = self.pos_encoding(x)
        x = self.encoder1(x)
        x = self.classifier(x)

        return x

In [ ]:
depth_model = Ranker1Layer()

train_model(depth_model,train_loader,val_loader)
depth_acc = evaluate(depth_model,test_loader)

print("1 Layer Transformer Test Acc:",depth_acc)